# Skewness in depth
Sample skewness by hand, the order of mode, median and mean, and reading skewness values on the Titanic data.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy import stats

## 1. Sample skewness by hand

In [ ]:
x = np.array([1, 2, 3, 4, 10])
n = len(x)
s = x.std(ddof=1)                     # sample standard deviation (n - 1)
z = (x - x.mean()) / s
G1 = n / ((n - 1) * (n - 2)) * (z ** 3).sum()
print("G1 by hand:", round(G1, 3))
print("pandas skew:", round(pd.Series(x).skew(), 3))
# without the correction: scipy's default
print("g1 (scipy):", round(stats.skew(x), 3))
# Pearson's coefficient
print("Pearson:", round(3 * (x.mean() - np.median(x)) / s, 3))

## 2. Mode, median and mean in skewed data

In [ ]:
df = pd.read_csv("data/titanic_train.csv")
fare = df["Fare"]
print("fare: mode", fare.mode()[0], "median", fare.median(), "mean", round(fare.mean(), 2))
# easy-exam marks: simulated left-skewed data
rng = np.random.default_rng(42)
marks = np.round(100 * rng.beta(5, 1.5, 1000), 1)
grid = np.linspace(0, 100, 4001)
mode_marks = grid[np.argmax(stats.gaussian_kde(marks)(grid))]   # peak of the KDE
print("marks: mean", round(marks.mean(), 1), "median", np.median(marks), "mode", round(mode_marks, 1))

In [ ]:
fig = go.Figure(go.Histogram(x=fare[fare < 150], nbinsx=30))
for name, v in [("mode", fare.mode()[0]), ("median", fare.median()), ("mean", fare.mean())]:
    fig.add_vline(x=v, annotation_text=name)
fig.update_layout(template="simple_white", xaxis_title="fare")
fig.show()

## 3. Reading skewness values

In [ ]:
def reading(value):
    if abs(value) < 0.5:
        return "approximately symmetric"
    if abs(value) <= 1:
        return "moderately skewed"
    return "highly skewed"

for col in ["Age", "Fare", "SibSp", "Parch"]:
    value = df[col].skew()
    print(f"{col:6s} {value:6.2f}  {reading(value)}")

## 4. Symmetric is not the same as normal

In [ ]:
uni = rng.uniform(0, 1, 10_000)
two_humps = np.r_[rng.normal(-3, 1, 5000), rng.normal(3, 1, 5000)]
print("uniform skew:", round(stats.skew(uni), 3), " two humps skew:", round(stats.skew(two_humps), 3))